# Getting Started with Tissot

This notebook demonstrates the core Tissot workflow: X-Ray, Check, Score, and Fix.

## Install

```bash
pip install tissot
```

In [ ]:
import json
import subprocess

def tissot(command: str, file: str, **kwargs) -> dict:
    """Run a tissot command and return JSON output."""
    cmd = ["tissot", command, file, "--json"]
    for key, value in kwargs.items():
        if isinstance(value, bool) and value:
            cmd.append(f"--{key}")
        elif not isinstance(value, bool):
            cmd.extend([f"--{key}", str(value)])
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return json.loads(result.stdout)

## 1. Projection X-Ray

Analyze projection distortion on a Web Mercator dataset.

In [ ]:
xray = tissot("xray", "../datasets/us_states_mercator.geojson", recommend=True)

print(f"CRS: {xray.get('crs', 'Unknown')}")
print(f"Mean area distortion: {xray['distortion']['mean_area_pct']:.2f}%")
print(f"Max area distortion: {xray['distortion']['max_area_pct']:.2f}%")
print(f"\nSample points: {xray.get('sample_count', 0)}")

for i, rec in enumerate(xray.get('recommendations', [])[:3], 1):
    print(f"\nRecommendation {i}: {rec['epsg']} ({rec.get('name', '')})")
    print(f"  Area distortion: {rec.get('mean_area_pct', 0):.2f}%")

## 2. Data Quality Check

Run all diagnostic rules on a dataset with known issues.

In [ ]:
check = tissot("check", "../datasets/parcels_with_issues.geojson")

summary = check['summary']
print(f"Total findings: {summary['total']}")
print(f"  Errors:   {summary['errors']}")
print(f"  Warnings: {summary['warnings']}")
print(f"  Info:     {summary['info']}")

print("\nFindings:")
for f in check['findings']:
    print(f"  [{f['severity']}] {f['rule_id']}: {f['message']}")

## 3. Quality Score

Get a Lighthouse-style quality rating.

In [ ]:
score = tissot("score", "../datasets/parcels_with_issues.geojson")

print(f"Overall: {score['overall_score']}/100 (Grade: {score['grade']})")
print("\nCategories:")
for name, cat in score.get('categories', {}).items():
    cat_score = cat['score'] if isinstance(cat, dict) else cat
    print(f"  {name}: {cat_score}/100")

## 4. Autofix

Reproject data to a better CRS.

In [ ]:
fix = tissot("fix", "../datasets/us_states_mercator.geojson", reproject="EPSG:5070")

print(f"Input:  {fix['input']}")
print(f"Output: {fix['output']}")
print(f"Updated features: {fix['updated_features']}")
for action in fix.get('actions', []):
    print(f"  - {action}")

## 5. Compare Before/After

Diff the original and fixed datasets.

In [ ]:
diff = tissot("diff", "../datasets/us_states_mercator.geojson")
# Note: diff requires two files — this is a placeholder showing the API pattern
print(json.dumps(diff, indent=2))